# Analisis de Finanzas y Riesgo Crediticio

Proyecto: Banca Atlas

Equip_30

Equipo: Finanzas y Riesgo Crediticio 

Mariia Zaitseva

_____

Semana: 3

_____

## Pregunta de negocio
Analistas de Finanzas y Riesgo Crediticio: ¿Qué umbrales de saldo podrían indicar más riesgo de morosidad?

### Imports

In [1]:
# !pip install optbinning

In [2]:
import pandas as pd
import numpy as np
from optbinning import OptimalBinning
import warnings

# graficos
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
pio.templates.default = "plotly_dark"

pio.templates["custom"] = pio.templates["plotly_dark"]
pio.templates["custom"].layout.paper_bgcolor = "#050a30"
pio.templates["custom"].layout.plot_bgcolor  = "#050a30"
pio.templates.default = "custom"

In [4]:
warnings.filterwarnings("ignore")

### Carga de datos

In [5]:
df = pd.read_csv("../../Data/06-08-2026/06-08-2026_Clean.csv", index_col=False, encoding='utf-8')

In [6]:
df

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,1,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,no_campaign,1
1,2,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,no_campaign,1
2,3,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,no_campaign,1
3,4,55,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,no_campaign,1
4,5,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,673,2,-1,0,no_campaign,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,11158,33,blue-collar,single,primary,0,1,1,0,cellular,20,apr,257,1,-1,0,no_campaign,0
11158,11159,39,services,married,secondary,0,733,0,0,unknown,16,jun,83,4,-1,0,no_campaign,0
11159,11160,32,technician,single,secondary,0,29,0,0,cellular,19,aug,156,2,-1,0,no_campaign,0
11160,11161,43,technician,married,secondary,0,0,0,1,cellular,8,may,9,2,172,5,failure,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         11162 non-null  int64 
 1   age        11162 non-null  int64 
 2   job        11162 non-null  object
 3   marital    11162 non-null  object
 4   education  11162 non-null  object
 5   default    11162 non-null  int64 
 6   balance    11162 non-null  int64 
 7   housing    11162 non-null  int64 
 8   loan       11162 non-null  int64 
 9   contact    11162 non-null  object
 10  day        11162 non-null  int64 
 11  month      11162 non-null  object
 12  duration   11162 non-null  int64 
 13  campaign   11162 non-null  int64 
 14  pdays      11162 non-null  int64 
 15  previous   11162 non-null  int64 
 16  poutcome   11162 non-null  object
 17  deposit    11162 non-null  int64 
dtypes: int64(12), object(6)
memory usage: 1.5+ MB


## Analisis

El metodo - **Optimal Binning**: el modelo divide las observaciones de “balance” a unos segmentos y busca la tasa de “default” y “no default” en cada segmento, encuentra el mejor rango que prediga “default”. 

### Preparar los datos y variables

X_var - la variable continua independente ('balance') 
y_var - la variable categorica objetiva ('default')

In [8]:
X_var = 'balance'
y_var = 'default'

In [9]:
X = df[X_var].values
y = df[y_var].values

### Entrenar el modelo

In [10]:
optb = OptimalBinning(
    name=X_var, dtype='numerical', solver='cp', monotonic_trend='auto'
)

optb.fit(X, y)

,name,'balance'
,dtype,'numerical'
,prebinning_method,'cart'
,solver,'cp'
,divergence,'iv'
,max_n_prebins,20
,min_prebin_size,0.05
,min_n_bins,None
,max_n_bins,None
,min_bin_size,None
,max_bin_size,None


La tabla de los resultados del modelo

In [11]:
binning_table = optb.binning_table
df_report = binning_table.build()

In [12]:
df_report

,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, -0.50)",688,0.061638,610,78,0.113372,-2.124391,0.868453,0.091865
1,"[-0.50, 17.50)",1086,0.097294,1048,38,0.034991,-0.864088,0.113080,0.013711
2,"[17.50, 86.50)",690,0.061817,672,18,0.026087,-0.561254,0.025828,0.003187
3,"[86.50, 162.50)",686,0.061459,680,6,0.008746,0.549192,0.014355,0.001772
4,"[162.50, 339.50)",1327,0.118886,1316,11,0.008289,0.603316,0.032715,0.004028
5,"[339.50, 457.50)",651,0.058323,648,3,0.004608,1.194137,0.049060,0.005792
6,"[457.50, 1238.50)",2488,0.222899,2477,11,0.004421,1.235767,0.197511,0.023229
7,"[1238.50, 1579.50)",560,0.050170,559,1,0.001786,2.145009,0.096297,0.010157
8,"[1579.50, 3157.50)",1487,0.133220,1486,1,0.000672,3.122702,0.403491,0.036573
9,"[3157.50, inf)",1499,0.134295,1498,1,0.000667,3.130745,0.407948,0.036928


Las columnas de la tabla:
- **Bin** - los valores umbral que dividen la variable 'balance' a los intervalos (bins).
- **Count** - la cantidad de los clientes en cada intervalo.
- **Count (%)** - el porcentaje de los clientes en cada intervalo.
- **Non-event** - la cantidad de "default=0" (no hay impago).
- **Event** - la cantidad de "default=1" (hay impago).
- **Event rate** - la tasa de default de todos los casos (observaciones).
- **WoE** (Weight of Evidence) - el principal indicador de riesgo del grupo. Un WoE positivo significa que los clientes en este intervalo son más fiables que el promedio de la muestra. Un WoE negativo indica un mayor riesgo de impago. El punto en el que el WoE cruza bruscamente el cero y cae es el umbral ideal para dar de baja a los clientes de riesgo.
- **IV** (Information Value) - el poder predictivo general de la característica. Si el IV es mayor que 0,1, la característica tiene una fuerza media; si es mayor que 0,3, tiene una fuerza alta.
- **JS** (Jensen-Shannon divergence) - muestra la contribución de cada intervalo específico (bin) al poder de separación general de la característica. Un valor JS alto (> 0,05 o 0,1) contribuye significativamente a la precisión del modelo.

**Conclusion**:

La columna 'WoE' de la tabla muestra que los tres primeros intervalos (bins) tienen un mayor riesgo de impago ('WoE' < 0).

#### La prueba por Bootstrap

Seleccionar umbrales básicos de los intervalos en los datos iniciales:

In [13]:
base_optb = OptimalBinning(
    name=X_var, dtype='numerical', solver='cp', monotonic_trend='auto'
)
base_optb.fit(df[X_var].values, df[y_var].values)
base_splits = base_optb.splits
print("Uumbrales básicos de los intervalos en los datos iniciales:")
base_splits.tolist()

Uumbrales básicos de los intervalos en los datos iniciales:


[-0.5, 17.5, 86.5, 162.5, 339.5, 457.5, 1238.5, 1579.5, 3157.5]

Bootstrap (el parametro 'user_splits' usa los datos del modelo 'base_splits'):

In [14]:
n_iterations = 200
all_bootstrap_data = []

for i in range(n_iterations):
    boot_df = df.sample(n=len(df), replace=True)

    if boot_df[y_var].nunique() < 2:
        continue

    boot_optb = OptimalBinning(
        name=X_var, dtype='numerical', solver='cp', user_splits=base_splits
    )
    boot_optb.fit(boot_df[X_var].values, boot_df[y_var].values)

    boot_report = boot_optb.binning_table.build()

    boot_intervals = boot_report[
        ~boot_report['Bin'].isin(['Special', 'Missing', 'Absolute always'])
    ].head(-1)

    for _, row in boot_intervals.iterrows():
        all_bootstrap_data.append(
            {
                'Iteration': i,
                'Bin': row['Bin'],
                'WoE': pd.to_numeric(row['WoE'], errors='coerce'),
            }
        )

df_boot_results = pd.DataFrame(all_bootstrap_data)

df_boot_results['WoE'] = df_boot_results['WoE'].replace(
    [np.inf, -np.inf], np.nan
)

Calcular intervalos de confianza:

In [15]:
summary_data = []

ordered_bins = base_optb.binning_table.build()['Bin'].head(-3).values

for bin_name in ordered_bins:
    if bin_name not in df_boot_results['Bin'].unique():
        continue
        
    group = df_boot_results[df_boot_results['Bin'] == bin_name]
    bin_woes = group['WoE'].dropna().values

    if len(bin_woes) == 0:
        continue

    low_ci = np.percentile(bin_woes, 2.5)
    high_ci = np.percentile(bin_woes, 97.5)
    mean_woe = np.mean(bin_woes)

    summary_data.append(
        {
            'Bin': bin_name,
            'Mean WoE': mean_woe,
            '95% CI low': low_ci,
            '95% CI high': high_ci,
            'Lower error': mean_woe - low_ci,
            'Upper error': high_ci - mean_woe,
        }
    )

df_bootstrap_summary = pd.DataFrame(summary_data)

df_bootstrap_summary.round(3)


,Bin,Mean WoE,95% CI low,95% CI high,Lower error,Upper error
0,"(-inf, -0.50)",-2.120,-2.267,-1.936,0.147,0.184
1,"[-0.50, 17.50)",-0.904,-1.129,-0.670,0.225,0.234
2,"[17.50, 86.50)",-0.503,-0.865,-0.004,0.362,0.499
3,"[86.50, 162.50)",0.341,-0.058,0.793,0.399,0.452
4,"[162.50, 339.50)",0.718,0.261,1.129,0.457,0.410
5,"[339.50, 457.50)",1.007,0.522,1.450,0.485,0.443
6,"[457.50, 1238.50)",1.346,0.940,1.882,0.406,0.536
7,"[1238.50, 1579.50)",2.043,1.390,2.300,0.654,0.257
8,"[1579.50, 3157.50)",2.692,2.179,3.132,0.513,0.441
9,"[3157.50, inf)",3.030,2.400,3.314,0.630,0.283


La visualización de los resultados de Bootstrap:

In [16]:
df_plot_woe = df_report[
    ~df_report["Bin"].isin(["Special", "Missing", "Absolute always"])
].head(-1).copy()

df_plot_woe['WoE'] = df_plot_woe['WoE'].astype('float64')

In [17]:
fig = go.Figure()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=df_bootstrap_summary['Bin'],
        y=df_bootstrap_summary['Mean WoE'],
        mode='markers+lines+text',
        name='Mean WoE',
        marker=dict(size=10, color='#2ecc71'),
        line=dict(dash='dash', color='#444444'),
        error_y=dict(
            type='data',
            symmetric=False,
            array=df_bootstrap_summary['Upper error'],
            arrayminus=df_bootstrap_summary['Lower error'],
            color='#e74c3c',
            thickness=2,
            width=6,
        ),
        text=df_plot_woe["WoE"].round(2),
        textposition="middle right",
        textfont=dict(color="#FFFFFF", size=11),
        hovertemplate="Intervalo: %{x}<br>WoE: %{y:.2f}<extra></extra>"
    )
)

fig.add_trace(
    go.Bar(
        x=df_plot_woe['Bin'],
        y=df_plot_woe['Count'],
        name="Número de clientes",
        marker_color='rgba(100, 110, 120, 0.35)',
        marker_line=dict(color='rgba(255, 255, 255, 0.15)', width=0.5),
        hovertemplate="Intervalo: %{x}<br>Clients: %{y}<extra></extra>"
    ),
    secondary_y=True
)

fig.update_layout(
    title="<b>Estabilidad de los intervalos WoE en el bootstrapping (equilibrio creciente)</b>",
    xaxis_title="Límites del intervalo de la variable independiente ('balance')",
    yaxis_title="WoE (Weight of Evidence)",
    
    xaxis=dict(
        type='category',
        categoryorder='array',
        categoryarray=df_bootstrap_summary['Bin'].tolist(),
        tickangle=-45
    ),

        yaxis2=dict(
        title="",
        showgrid=False,
        tickfont=dict(color="#B0B0B0", size=12),
        title_font=dict(color="#B0B0B0", size=12),
        overlaying="y",
        side="right"
    ),

    template='custom',
    height=550,
    width=900,
)

fig.show()

**Conclusión**: 

Barras de error de los dos primeros intervalos y el último muestran alta estabilidad de la evaluación de riesgos. La mayor parte de los impagos se concentran en las primeras dos zonas y casi no hay default en el fin. Los rangos intermedios (de 17,5 a 3157,5) son prácticamente idénticos en cuanto al nivel de riesgo. El modelo no puede demostrar de forma fiable que un cliente con un saldo de 50 euro sea más fiable que uno con un saldo de 200 euro.
Es necesario ampliar manualmente los rangos combinando los intervalos cuyos barras de error se superponen significativamente.

### Ajustificación del modelo, 5 bins

Crear los intervalos manualmente [-inf, -0.5), [-0.5, 17.5), [17.5, 339.5), [339.5, 1579.5), [1579.5, inf):

In [18]:
bins5_splits = [-0.5, 17.5, 339.5, 1579.5]

Entrenar el modelo:

In [19]:
bins5_optb = OptimalBinning(
    name=X_var, 
    dtype='numerical', 
    solver='cp', 
    user_splits=bins5_splits
)

bins5_optb.fit(df[X_var].values, df[y_var].values)

,name,'balance'
,dtype,'numerical'
,prebinning_method,'cart'
,solver,'cp'
,divergence,'iv'
,max_n_prebins,20
,min_prebin_size,0.05
,min_n_bins,None
,max_n_bins,None
,min_bin_size,None
,max_bin_size,None


La tabla de los resultados del modelo

In [20]:
bins5_report = bins5_optb.binning_table.build()

bins5_report.round(4)

,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, -0.50)",688,0.0616,610,78,0.1134,-2.124391,0.8685,0.0919
1,"[-0.50, 17.50)",1086,0.0973,1048,38,0.0350,-0.864088,0.1131,0.0137
2,"[17.50, 339.50)",2703,0.2422,2668,35,0.0129,0.152595,0.0052,0.0007
3,"[339.50, 1579.50)",3699,0.3314,3684,15,0.0041,1.322563,0.3251,0.0379
4,"[1579.50, inf)",2986,0.2675,2984,2,0.0007,3.126732,0.8114,0.0735
5,Special,0,0.0000,0,0,0.0000,0.0,0.0000,0.0000
6,Missing,0,0.0000,0,0,0.0000,0.0,0.0000,0.0000
Totals,,11162,1.0000,10994,168,0.0151,,2.1233,0.2176


Los resultados en la tabla:
- **Bin** - los valores umbral que dividen la variable 'balance' a 5 intervalos (bins) más grandes.
- **Count**, **Count (%)** - la cantidad de los clientes en cada intervalo está suficiente.
- **Non-event**, **Event** - en cada intervalo hay suficiente "default=0" (no hay impago) y "default=1" (hay impago).
- **Event rate** - la tasa de default es diferente en cada intervalo.
- **WoE** (Weight of Evidence) - los dos primeros intervalos muestran WoE negativo que indica un mayor riesgo de impago (más de 0,5). 
- **IV** (Information Value), **JS** (Jensen-Shannon divergence) - muestran el poder predictivo general de las características y del modelo. 

#### Bootstrap del modelo ajustificado, 5 bins

In [21]:
n_iterations = 200
bins5_bootstrap_data = []


for i in range(n_iterations):
    boot_df = df.sample(n=len(df), replace=True)

    if boot_df[y_var].nunique() < 2:
        continue

    boot_optb = OptimalBinning(
        name=X_var, dtype='numerical', solver='cp', user_splits=bins5_splits
    )
    boot_optb.fit(boot_df[X_var].values, boot_df[y_var].values)


    boot_report = boot_optb.binning_table.build()

    boot_intervals = boot_report[
        ~boot_report['Bin'].isin(['Special', 'Missing', 'Absolute always'])
    ].head(-1)

    for _, row in boot_intervals.iterrows():
        bins5_bootstrap_data.append(
            {
                'Iteration': i,
                'Bin': row['Bin'],
                'WoE': pd.to_numeric(row['WoE'], errors='coerce'),
            }
        )


df_bins5_boot_results = pd.DataFrame(bins5_bootstrap_data)
df_bins5_boot_results['WoE'] = df_bins5_boot_results['WoE'].replace(
    [np.inf, -np.inf], np.nan
)

base_optb = OptimalBinning(
    name=X_var, dtype='numerical', solver='cp', user_splits=bins5_splits
)
base_optb.fit(df[X_var].values, df[y_var].values)
ordered_bins = base_optb.binning_table.build()['Bin'].head(-3).values

Calcular de intervalos de confianza:

In [22]:
bins5_summary_data = []

for bin_name in ordered_bins:
    if bin_name not in df_bins5_boot_results['Bin'].unique():
        continue

    group = df_bins5_boot_results[df_bins5_boot_results['Bin'] == bin_name]
    bin_woes = group['WoE'].dropna().values

    if len(bin_woes) == 0:
        continue

    low_ci = np.percentile(bin_woes, 2.5)
    high_ci = np.percentile(bin_woes, 97.5)
    mean_woe = np.mean(bin_woes)

    bins5_summary_data.append(
        {
            'Bin': bin_name,
            'Mean WoE': mean_woe,
            '95% CI low': low_ci,
            '95% CI high': high_ci,
            'Lower error': mean_woe - low_ci,
            'Upper error': high_ci - mean_woe,
        }
    )

df_bootstrap_bins5_summary = pd.DataFrame(bins5_summary_data)

df_bootstrap_bins5_summary.round(3)

,Bin,Mean WoE,95% CI low,95% CI high,Lower error,Upper error
0,"(-inf, -0.50)",-2.131,-2.313,-1.932,0.182,0.198
1,"[-0.50, 17.50)",-0.851,-1.104,-0.503,0.253,0.348
2,"[17.50, 339.50)",0.165,-0.115,0.485,0.280,0.320
3,"[339.50, 1579.50)",1.371,0.967,1.861,0.403,0.491
4,"[1579.50, inf)",3.148,2.165,3.894,0.983,0.746


La visualización de los resultados de Bootstrap ajustificado:

In [23]:
df_plot_woe_bins5 = bins5_report[
    ~bins5_report["Bin"].isin(["Special", "Missing", "Absolute always"])
].head(-1).copy()

df_plot_woe_bins5['WoE'] = df_plot_woe_bins5['WoE'].astype('float64')

In [24]:
fig = go.Figure()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=df_bootstrap_bins5_summary['Bin'],
        y=df_bootstrap_bins5_summary['Mean WoE'],
        mode='markers+lines+text',
        name='Mean WoE',
        marker=dict(size=10, color='#2ecc71'),
        line=dict(dash='dash', color='#444444'),
        error_y=dict(
            type='data',
            symmetric=False,
            array=df_bootstrap_bins5_summary['Upper error'],
            arrayminus=df_bootstrap_bins5_summary['Lower error'],
            color='#e74c3c',
            thickness=2,
            width=6,
        ),
        text=df_plot_woe_bins5["WoE"].round(2),
        textposition="middle right",
        textfont=dict(color="#FFFFFF", size=11),
        hovertemplate="Intervalo: %{x}<br>WoE: %{y:.2f}<extra></extra>"
    )
)

fig.add_trace(
    go.Bar(
        x=df_plot_woe_bins5['Bin'],
        y=df_plot_woe_bins5['Count'],
        name="Número de clientes",
        marker_color='rgba(100, 110, 120, 0.35)',
        marker_line=dict(color='rgba(255, 255, 255, 0.15)', width=0.5),
        hovertemplate="Intervalo: %{x}<br>Clients: %{y}<extra></extra>"
    ),
    secondary_y=True
)

fig.update_layout(
    title="<b>Estabilidad de los intervalos WoE en el bootstrapping (equilibrio creciente)</b>",
    xaxis_title="Límites del intervalo de la variable independiente ('balance')",
    yaxis_title="WoE (Weight of Evidence)",
    
    xaxis=dict(
        type='category',
        categoryorder='array',
        categoryarray=df_bootstrap_bins5_summary['Bin'].tolist(),
        tickangle=-45
    ),

    yaxis2=dict(
        title="",
        showgrid=False,
        tickfont=dict(color="#B0B0B0", size=12),
        title_font=dict(color="#B0B0B0", size=12),
        overlaying="y",
        side="right"
    ),

    template='custom',
    height=550,
    width=900,
)

fig.show()

**Conclusión**: 
Barras de error de los intervalos no se cruzan que muestra alta estabilidad de la evaluación de riesgos. La mayor parte de los impagos se concentran en las primeras dos zonas y casi no hay default en el fin. Los rangos están bien ivididos en cuanto al nivel de riesgo. El modelo puede demostrar de forma fiable la diferencia entre los clientes de varios intervalos, que los clientes de un segmento es consistentemente más fiable que el anterior y consistentemente más riesgoso que el siguiente.

### Ajustificación del modelo, 4 bins

Crear los intervalos manualmente [-inf, -0.5), [-0.5, 17.5), [17.5, 1415.0), [1415.0, inf):

In [25]:
bins4_splits = [-0.5, 17.5, 1415.0]

Entrenar el modelo:

In [26]:
bins4_optb = OptimalBinning(
    name=X_var, 
    dtype='numerical', 
    solver='cp', 
    user_splits=bins4_splits
)

bins4_optb.fit(df[X_var].values, df[y_var].values)

,name,'balance'
,dtype,'numerical'
,prebinning_method,'cart'
,solver,'cp'
,divergence,'iv'
,max_n_prebins,20
,min_prebin_size,0.05
,min_n_bins,None
,max_n_bins,None
,min_bin_size,None
,max_bin_size,None


La tabla de resultados del modelo

In [27]:
bins4_report = bins4_optb.binning_table.build()

bins4_report.round(4)

,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, -0.50)",688,0.0616,610,78,0.1134,-2.124391,0.8685,0.0919
1,"[-0.50, 17.50)",1086,0.0973,1048,38,0.0350,-0.864088,0.1131,0.0137
2,"[17.50, 1415.00)",6176,0.5533,6126,50,0.0081,0.627133,0.1628,0.0200
3,"[1415.00, inf)",3212,0.2878,3210,2,0.0006,3.199738,0.8962,0.0802
4,Special,0,0.0000,0,0,0.0000,0.0,0.0000,0.0000
5,Missing,0,0.0000,0,0,0.0000,0.0,0.0000,0.0000
Totals,,11162,1.0000,10994,168,0.0151,,2.0405,0.2058


Los resultados en la tabla:
- **Bin** - los valores umbral que dividen la variable 'balance' a 4 intervalos (bins) más grandes.
- **Count**, **Count (%)** - la cantidad de los clientes en cada intervalo está suficiente.
- **Non-event**, **Event** - en cada intervalo hay suficiente "default=0" (no hay impago) y "default=1" (hay impago).
- **Event rate** - la tasa de default es diferente en cada intervalo.
- **WoE** (Weight of Evidence) - los dos primeros intervalos muestran WoE negativo que indica un mayor riesgo de impago (más de 0,5). 
- **IV** (Information Value), **JS** (Jensen-Shannon divergence) - muestran el poder predictivo general de las características y del modelo. 

#### Bootstrap del modelo ajustificado, 4 bins

In [28]:
n_iterations = 200
bins4_bootstrap_data = []


for i in range(n_iterations):
    boot_df = df.sample(n=len(df), replace=True)

    if boot_df[y_var].nunique() < 2:
        continue

    boot_optb = OptimalBinning(
        name=X_var, dtype='numerical', solver='cp', user_splits=bins4_splits
    )
    boot_optb.fit(boot_df[X_var].values, boot_df[y_var].values)


    boot_report = boot_optb.binning_table.build()

    boot_intervals = boot_report[
        ~boot_report['Bin'].isin(['Special', 'Missing', 'Absolute always'])
    ].head(-1)

    for _, row in boot_intervals.iterrows():
        bins4_bootstrap_data.append(
            {
                'Iteration': i,
                'Bin': row['Bin'],
                'WoE': pd.to_numeric(row['WoE'], errors='coerce'),
            }
        )


df_bins4_boot_results = pd.DataFrame(bins4_bootstrap_data)
df_bins4_boot_results['WoE'] = df_bins4_boot_results['WoE'].replace(
    [np.inf, -np.inf], np.nan
)

base_optb = OptimalBinning(
    name=X_var, dtype='numerical', solver='cp', user_splits=bins4_splits
)
base_optb.fit(df[X_var].values, df[y_var].values)
ordered_bins = base_optb.binning_table.build()['Bin'].head(-3).values

Calcular de intervalos de confianza:

In [29]:
bins4_summary_data = []

for bin_name in ordered_bins:
    if bin_name not in df_bins4_boot_results['Bin'].unique():
        continue

    group = df_bins4_boot_results[df_bins4_boot_results['Bin'] == bin_name]
    bin_woes = group['WoE'].dropna().values

    if len(bin_woes) == 0:
        continue

    low_ci = np.percentile(bin_woes, 2.5)
    high_ci = np.percentile(bin_woes, 97.5)
    mean_woe = np.mean(bin_woes)

    bins4_summary_data.append(
        {
            'Bin': bin_name,
            'Mean WoE': mean_woe,
            '95% CI low': low_ci,
            '95% CI high': high_ci,
            'Lower error': mean_woe - low_ci,
            'Upper error': high_ci - mean_woe,
        }
    )

df_bootstrap_bins4_summary = pd.DataFrame(bins4_summary_data)

df_bootstrap_bins4_summary.round(3)

,Bin,Mean WoE,95% CI low,95% CI high,Lower error,Upper error
0,"(-inf, -0.50)",-2.122,-2.279,-1.934,0.157,0.189
1,"[-0.50, 17.50)",-0.849,-1.099,-0.557,0.250,0.292
2,"[17.50, 1415.00)",0.635,0.438,0.849,0.197,0.214
3,"[1415.00, inf)",3.240,2.209,3.989,1.030,0.749


La visualización de los resultados de Bootstrap ajustificado:

In [30]:
df_plot_woe_bins4 = bins4_report[
    ~bins4_report["Bin"].isin(["Special", "Missing", "Absolute always"])
].head(-1).copy()

df_plot_woe_bins4['WoE'] = df_plot_woe_bins4['WoE'].astype('float64')

In [31]:
fig = go.Figure()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=df_bootstrap_bins4_summary['Bin'],
        y=df_bootstrap_bins4_summary['Mean WoE'],
        mode='markers+lines+text',
        name='Mean WoE',
        marker=dict(size=10, color='#2ecc71'),
        line=dict(dash='dash', color='#444444'),
        error_y=dict(
            type='data',
            symmetric=False,
            array=df_bootstrap_bins4_summary['Upper error'],
            arrayminus=df_bootstrap_bins4_summary['Lower error'],
            color='#e74c3c',
            thickness=2,
            width=6,
        ),
        text=df_plot_woe_bins4["WoE"].round(2),
        textposition="middle right",
        textfont=dict(color="#FFFFFF", size=11),
        hovertemplate="Intervalo: %{x}<br>WoE: %{y:.2f}<extra></extra>"
    )
)

fig.add_trace(
    go.Bar(
        x=df_plot_woe_bins4['Bin'],
        y=df_plot_woe_bins4['Count'],
        name="Número de clientes",
        marker_color='rgba(100, 110, 120, 0.35)',
        marker_line=dict(color='rgba(255, 255, 255, 0.15)', width=0.5),
        hovertemplate="Intervalo: %{x}<br>Clients: %{y}<extra></extra>"
    ),
    secondary_y=True
)


fig.update_layout(
    title="<b>Estabilidad de los intervalos WoE en el bootstrapping (equilibrio creciente)</b>",
    xaxis_title="Límites del intervalo de la variable independiente ('balance')",
    yaxis_title="WoE (Weight of Evidence)",
    
    xaxis=dict(
        type='category',
        categoryorder='array',
        categoryarray=df_bootstrap_bins4_summary['Bin'].tolist(),
        tickangle=-45
    ),

    yaxis2=dict(
        title="",
        showgrid=False,
        tickfont=dict(color="#B0B0B0", size=12),
        title_font=dict(color="#B0B0B0", size=12),
        overlaying="y",
        side="right"
    ),

    template='custom',
    height=550,
    width=900,
)

fig.show()

**Conclusión**: 

Barras de error de los intervalos muestran alta estabilidad de la evaluación de riesgos. El cuarto interval incluse a todos los clientes con saldo más de 1415 euro y pocos impagos. La mayor parte de los impagos se concentran en las primeras dos zonas y casi no hay default en el fin. Pero la división a cinco intervalos ofrece más opciónes para dividir a los clientes por grupos por el saldo.

### Saldo bajo

**Conclusión**: 

El umbral de saldo bajo se puede considerar de 17.50 euro.

Asignar un coeficiente de riesgo a los clientes en función de su saldo. Este coeficiente se puede utilizar para un modelo logistico para prediciones de impago.

In [32]:
df["balance_WoE"] = bins5_optb.transform(df["balance"].values, metric="woe")

df

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,balance_WoE
0,1,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,no_campaign,1,3.126732
1,2,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,no_campaign,1,0.152595
2,3,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,no_campaign,1,1.322563
3,4,55,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,no_campaign,1,3.126732
4,5,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,673,2,-1,0,no_campaign,1,0.152595
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,11158,33,blue-collar,single,primary,0,1,1,0,cellular,20,apr,257,1,-1,0,no_campaign,0,-0.864088
11158,11159,39,services,married,secondary,0,733,0,0,unknown,16,jun,83,4,-1,0,no_campaign,0,1.322563
11159,11160,32,technician,single,secondary,0,29,0,0,cellular,19,aug,156,2,-1,0,no_campaign,0,0.152595
11160,11161,43,technician,married,secondary,0,0,0,1,cellular,8,may,9,2,172,5,failure,0,-0.864088


In [33]:
fig = go.Figure()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=df_plot_woe_bins5['Bin'],
        y=df_plot_woe_bins5['Count'],
        name="Número de clientes",
        marker_color='rgba(100, 110, 120, 0.55)',
        marker_line=dict(color='rgba(0, 255, 255, 0.55)', width=0.5),
        hovertemplate="Intervalo: %{x}<br>Clients: %{y}<extra></extra>"
    ),
    secondary_y=True
)

fig.add_trace(
    go.Scatter(
        x=df_bootstrap_bins5_summary['Bin'],
        y=df_bootstrap_bins5_summary['Mean WoE'],
        mode='markers+lines+text',
        name='Mean WoE',
        marker=dict(size=10, color='#2ecc71'),
        line=dict(dash='dash', color='#444444'),
        error_y=dict(
            type='data',
            symmetric=False,
            array=df_bootstrap_bins5_summary['Upper error'],
            arrayminus=df_bootstrap_bins5_summary['Lower error'],
            color='#e74c3c',
            thickness=2,
            width=6,
        ),
        text=df_plot_woe_bins5["WoE"].round(2),
        textposition="middle right",
        textfont=dict(color="#FFFFFF", size=11),
        hovertemplate="Intervalo: %{x}<br>WoE: %{y:.2f}<extra></extra>"
    ),
)

fig.update_layout(
    title="<b>Grupos de los clientes por riesgo de impago según su saldo</b>",
    xaxis_title="Los intervalos de saldo",
    yaxis_title="Coeficiente WoE de impago (Weight of Evidence)",
    
    xaxis=dict(
        type='category',
        categoryorder='array',
        categoryarray=df_bootstrap_bins5_summary['Bin'].tolist(),
        tickangle=-45
    ),

    yaxis2=dict(
        title="",
        showgrid=False,
        tickfont=dict(color="#B0B0B0", size=12),
        title_font=dict(color="#B0B0B0", size=12),
        overlaying="y",
        side="right"
    ),

    template='custom',
    height=550,
    width=900,
)

fig.show()

#### Número de clientes en el grupo de saldo bajo

In [34]:
df['low_balance'] = np.where(
    df['balance'] <= 17.50,
    1,
    0
)

El número de clientes en el grupo de saldo bajo es 1774.

In [35]:
counts = df['low_balance'].value_counts()
counts

low_balance
0    9388
1    1774
Name: count, dtype: int64

#### Proporción de impagos en el grupo de saldo bajo con respecto al resto:

El número de impagos entre el grupo de clientes con saldo bajo es de 116 casos, en comparación con 52 entre los demás clientes.

In [36]:
contingency = pd.crosstab(
    df['low_balance'],
    df['default']
)

contingency

default,0,1
low_balance,,
0,9336,52
1,1658,116


#### La probabilidad de incumplimiento

Los clientes con saldos bajos tienen 11,8 veces más probabilidades de incurrir en impago.

In [37]:
default_rates = (
    df.groupby('low_balance')['default']
    .mean()
    .reset_index(name='default_probability')
)

default_rates

,low_balance,default_probability
0,0,0.005539
1,1,0.065389


## RESUMEN

- El umbral de saldo bajo se puede considerar de 17.50 euro.

- El umbral que muestra el riesgo más alto de impago es -0.50 euro.

- El modelo puede demostrar de forma fiable la diferencia entre los clientes de varios intervalos, que los clientes de un segmento es consistentemente más fiable que el anterior y consistentemente más riesgoso que el siguiente.

- Dividir a los clientes a diferentes grupos por su nivel de saldo es una parte del sistema de coeficientes (scoring) de los clientes para definir y ofrecerles diferentes productos bancarios.

#### Recomendaciones para la política de riesgos

Respecto a los clientes clasificados con un alto riesgo de default, se recomienda endurecer la política crediticia mediante las siguientes acciones:

   - Implementar un sistema de seguimiento del saldo del cliente.

   - Establecer un saldo de cuenta mínimo obligatorio y un sistema de comisiones por incumplimiento.

   - Implementar un sistema de los coeficientes que valoran el nivel de saldo.

   - Implementar un sistema de los coeficientes que valoran corportamiento financiero del cliente (varios tipos de préstamos, sumas de préstamos, deposit, etc).
